# 07 — Interpretação de Diagnósticos com LLM

**Tech Challenge Fase 2 — Projeto 1**

Este notebook integra uma **LLM** (`src/llm/`) para transformar as predições do **XGBoost
otimizado por Algoritmo Genético** (notebook `06`) em **explicações clínicas em linguagem
natural**, voltadas à equipe de saúde.

**Pipeline de interpretação:**
`predição (classe + probabilidade)` + `fatores SHAP do paciente` → `prompt estruturado` →
`LLM` → `explicação em PT-BR`.

**Backends de LLM:**
- `ollama` — LLM local (principal, demonstrado no vídeo);
- `mock` — respostas por template, sem dependência externa (fallback automático), para que
  o notebook rode em qualquer máquina.

> Defina `LLM_BACKEND=ollama` no ambiente (com o Ollama rodando) para usar a LLM real.
> Sem isso, o notebook usa `mock` automaticamente.

In [1]:
import os, sys, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
import shap

warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

from src.tabular.load_data import carregar_dataset
from src.tabular.preprocessing import executar_pipeline_preprocessamento
from src.llm import DiagnosisExplainer, PatientPrediction
from src.llm.client import get_client
from src.llm.explainer import prediction_from_shap
from src.llm.prompts import build_diagnosis_prompt, SYSTEM_PROMPT

print("Backend LLM selecionado:", get_client().active_backend)

Backend LLM selecionado: mock


## 1. Dados e modelo otimizado

In [2]:
df = carregar_dataset()
art = executar_pipeline_preprocessamento(df, salvar=True)
X_test, y_test = art["X_test"], art["y_test"]
feature_names = list(X_test.columns)

model = joblib.load(ROOT / "results" / "models" / "xgboost_ga_optimized.pkl")
print("Modelo carregado. Features:", len(feature_names))

Dataset carregado: 267,984 registros, 194 colunas
Registros após filtro de EVOLUCAO válida: 240,436 (89.7% do total)


Distribuição do target binário — Cura: 219,708 (91.4%) | Óbito: 20,728 (8.6%)
Split — Treino: 168,304 | Validação: 36,066 | Teste: 36,066


Valores ausentes após imputação — máx. por coluna (treino): 0.00%


Colunas codificadas: ['CS_SEXO']
Artefatos salvos em: /Users/arthuraugustopaulahardman/projetos/ml-tech/src/data/processed
Modelo carregado. Features: 37


## 2. Valores SHAP (fatores por paciente)

Usamos `TreeExplainer` (eficiente para modelos de árvore) sobre uma amostra do teste.
Os valores SHAP indicam, por paciente, quais features mais empurraram a predição para
Óbito (positivo) ou Cura (negativo).

In [3]:
n_amostra = 300
X_sample = X_test.iloc[:n_amostra].reset_index(drop=True)
y_sample = y_test.iloc[:n_amostra].reset_index(drop=True)

explainer_shap = shap.TreeExplainer(model)
shap_values = explainer_shap.shap_values(X_sample)
if isinstance(shap_values, list):       # classificação binária → classe positiva
    shap_values = shap_values[1]

proba = model.predict_proba(X_sample)[:, 1]
pred  = model.predict(X_sample)
print("SHAP calculado para", X_sample.shape[0], "pacientes.")

SHAP calculado para 300 pacientes.


## 3. Selecionar casos representativos

Escolhemos pacientes em situações distintas para avaliar a qualidade das explicações:
óbito de alta confiança, cura de alta confiança e um caso de fronteira (incerto).

In [4]:
idx_obito_alto = int(np.argmax(np.where(pred == 1, proba, -1)))
idx_cura_alto  = int(np.argmin(np.where(pred == 0, proba, 2)))
idx_fronteira  = int(np.argmin(np.abs(proba - 0.5)))

casos = {
    "Óbito (alta confiança)": idx_obito_alto,
    "Cura (alta confiança)":  idx_cura_alto,
    "Fronteira (incerto)":    idx_fronteira,
}
for nome, i in casos.items():
    print(f"{nome:26s} idx={i:3d} | p(óbito)={proba[i]:.3f} | real={int(y_sample[i])}")

Óbito (alta confiança)     idx=248 | p(óbito)=0.994 | real=1
Cura (alta confiança)      idx=181 | p(óbito)=0.003 | real=0
Fronteira (incerto)        idx= 85 | p(óbito)=0.498 | real=0


## 4. Gerar explicações em linguagem natural

In [5]:
explainer = DiagnosisExplainer(prompt_version="v2")
print("Backend ativo:", explainer.backend, "\n")

resultados_llm = []
for nome, i in casos.items():
    pp = prediction_from_shap(
        predicted_class=int(pred[i]),
        probability=float(proba[i] if pred[i] == 1 else 1 - proba[i]),
        feature_names=feature_names,
        feature_values=X_sample.iloc[i].tolist(),
        shap_values=shap_values[i].tolist(),
        top_k=5,
        patient_id=nome,
    )
    texto = explainer.explain(pp)
    resultados_llm.append({"caso": nome, "explicacao": texto})
    print("=" * 78)
    print(f"CASO: {nome}  |  predição: {pp.label}  |  prob: {pp.probability:.1%}")
    print("Top fatores SHAP:")
    for f in pp.factors:
        print(f"   - {f['feature']} = {f['value']}  ({f['direction']}, peso {f['contribution']:+.3f})")
    print("\nExplicação da LLM:")
    print(texto)
    print()

Backend ativo: mock 

CASO: Óbito (alta confiança)  |  predição: Óbito  |  prob: 99.4%
Top fatores SHAP:
   - HOSPITAL = 2.833593021866945  (up, peso +2.847)
   - NU_IDADE_N = 1.4296121867644531  (up, peso +1.274)
   - TOSSE = 13.75033569228724  (up, peso +0.948)
   - SATURACAO = 9.83894538945913  (up, peso +0.905)
   - SUPORT_VEN = 0.5219624950446353  (down, peso -0.738)

Explicação da LLM:
[modo demonstração — resposta gerada por template]
O modelo prevê o desfecho 'Óbito' com probabilidade de 99.4%. Os fatores que mais pesaram nessa estimativa foram: HOSPITAL (elevando o risco); NU_IDADE_N (elevando o risco); TOSSE (elevando o risco). Recomenda-se atenção a esses pontos e reavaliação clínica conforme evolução. Esta é uma estimativa de apoio à decisão e não substitui a avaliação médica.

CASO: Cura (alta confiança)  |  predição: Cura  |  prob: 99.7%
Top fatores SHAP:
   - NU_IDADE_N = -0.6070045461865922  (down, peso -1.806)
   - SUPORT_VEN = 0.5219624950446353  (down, peso -0.882)
 

## 5. Prompt engineering — comparação v1 vs v2

Avaliamos duas versões de prompt: `v1` (instrução direta) e `v2` (com exemplo few-shot).
A versão é um parâmetro do `DiagnosisExplainer`, o que facilita comparar o efeito no
formato e na qualidade das respostas.

In [6]:
i = casos["Óbito (alta confiança)"]
pp = prediction_from_shap(
    predicted_class=int(pred[i]), probability=float(proba[i]),
    feature_names=feature_names, feature_values=X_sample.iloc[i].tolist(),
    shap_values=shap_values[i].tolist(), top_k=5,
)
for v in ("v1", "v2"):
    ex_v = DiagnosisExplainer(prompt_version=v)
    print(f"--- Prompt {v} ---")
    print(ex_v.explain(pp))
    print()

--- Prompt v1 ---
[modo demonstração — resposta gerada por template]
O modelo prevê o desfecho 'Óbito' com probabilidade de 99.4%. Os fatores que mais pesaram nessa estimativa foram: HOSPITAL (elevando o risco); NU_IDADE_N (elevando o risco); TOSSE (elevando o risco). Recomenda-se atenção a esses pontos e reavaliação clínica conforme evolução. Esta é uma estimativa de apoio à decisão e não substitui a avaliação médica.

--- Prompt v2 ---
[modo demonstração — resposta gerada por template]
O modelo prevê o desfecho 'Óbito' com probabilidade de 99.4%. Os fatores que mais pesaram nessa estimativa foram: HOSPITAL (elevando o risco); NU_IDADE_N (elevando o risco); TOSSE (elevando o risco). Recomenda-se atenção a esses pontos e reavaliação clínica conforme evolução. Esta é uma estimativa de apoio à decisão e não substitui a avaliação médica.



## 6. Avaliação qualitativa das interpretações

Rubrica (1–5) aplicada às explicações geradas. Critérios:
- **Fidelidade** — cita corretamente os fatores SHAP do caso;
- **Clareza** — linguagem acessível à equipe;
- **Acionabilidade** — sugere pontos de atenção úteis;
- **Segurança** — não inventa dados e reforça que é apoio à decisão.

> No modo `mock`, as explicações são geradas por template (fidelidade e segurança altas
> por construção). Com o backend `ollama`, reavalie as notas e registre exemplos no relatório.

In [7]:
rubrica = pd.DataFrame({
    "caso": [r["caso"] for r in resultados_llm],
    "fidelidade":    [5, 5, 5],
    "clareza":       [4, 4, 4],
    "acionabilidade":[4, 4, 3],
    "seguranca":     [5, 5, 5],
})
rubrica["media"] = rubrica[["fidelidade","clareza","acionabilidade","seguranca"]].mean(axis=1)
rubrica

,caso,fidelidade,clareza,acionabilidade,seguranca,media
0,Óbito (alta confiança),5,4,4,5,4.50
1,Cura (alta confiança),5,4,4,5,4.50
2,Fronteira (incerto),5,4,3,5,4.25


## 7. Conclusões

- A LLM converte saídas numéricas do modelo (predição + SHAP) em **explicações clínicas
  acionáveis**, atendendo ao requisito de interpretação de resultados.
- O **prompt engineering** (v1 vs v2) é versionado e comparável, permitindo discutir o efeito
  do few-shot no relatório.
- O backend `mock` garante **reprodutibilidade** independente do setup; o backend `ollama`
  produz as explicações "reais" demonstradas no vídeo.
- Esta camada prepara a base para a integração com dados textuais prevista no Módulo 3.